# Lab 2 CIFAR-10 Classification - Colab Runner

VS Code에서 코드를 수정한 뒤 GitHub에 push하고, 이 노트북에서는 `git pull` 후 Colab GPU로 학습을 실행합니다. 과제 조건 유지를 위해 `configs/*.yaml`은 직접 수정하지 않고, Colab 경고 방지를 위해 실행 시 `dataset.num_workers=2`만 override합니다.

In [5]:
!nvidia-smi

import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

Fri May 29 12:05:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. GitHub 프로젝트 가져오기

처음 실행할 때는 clone 셀을 사용합니다. 이미 clone되어 있으면 다음 셀의 `git pull`만 실행하면 됩니다.

In [6]:
%cd /content
!rm -rf lab2-problem
!git clone https://github.com/dodhdo23/lab2-problem.git
%cd /content/lab2-problem
!git status

/content
Cloning into 'lab2-problem'...
remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 24 (delta 8), reused 24 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (24/24), 10.36 KiB | 10.36 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/lab2-problem
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


## 2. 최신 코드 반영

VS Code에서 수정 후 `git push`했다면 Colab에서는 아래 셀을 실행해 최신 코드를 가져옵니다.

In [7]:
%cd /content/lab2-problem
!git pull
!git log --oneline -5
!git status

/content/lab2-problem
Already up to date.
cbdd308 (HEAD -> main, origin/main, origin/HEAD) initial commit
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


## 3. 패키지 설치

In [8]:
%cd /content/lab2-problem
!pip install -r requirements.txt

/content/lab2-problem


## 4. 1 epoch 테스트

전체 학습 전에 구현 오류를 빠르게 확인합니다.

In [9]:
%cd /content/lab2-problem

!python train.py --config configs/linear.yaml --opts train.epochs=1 dataset.num_workers=2
!python train.py --config configs/mlp_512.yaml --opts train.epochs=1 dataset.num_workers=2
!python train.py --config configs/mlp_2048.yaml --opts train.epochs=1 dataset.num_workers=2
!python train.py --config configs/convnet_v1.yaml --opts train.epochs=1 dataset.num_workers=2
!python train.py --config configs/convnet_v2.yaml --opts train.epochs=1 dataset.num_workers=2
!python train.py --config configs/convnet_v3_reg.yaml --opts train.epochs=1 dataset.num_workers=2
!python train.py --config configs/convnet_v3_aug.yaml --opts train.epochs=1 dataset.num_workers=2
!python train.py --config configs/resnet50_finetune.yaml --opts train.epochs=1 dataset.num_workers=2

/content/lab2-problem
2026-05-29 12:05:53,898 | INFO | Run directory: outputs/linear_20260529-120553
2026-05-29 12:05:53,898 | INFO | Device: cuda
100% 170M/170M [00:02<00:00, 79.3MB/s] 
Traceback (most recent call last):
  File "/content/lab2-problem/train.py", line 167, in <module>
    cli()
  File "/content/lab2-problem/train.py", line 163, in cli
    run_training(config)
  File "/content/lab2-problem/train.py", line 68, in run_training
    model = build_model(config.get("model", {})).to(device)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/lab2-problem/src/models.py", line 221, in build_model
    return LinearClassifier(
           ^^^^^^^^^^^^^^^^^
  File "/content/lab2-problem/src/models.py", line 39, in __init__
    raise NotImplementedError("Problem 1: implement LinearClassifier.__init__")
NotImplementedError: Problem 1: implement LinearClassifier.__init__
2026-05-29 12:06:02,967 | INFO | Run directory: outputs/mlp_512_20260529-120602
2026-05-29 12:06:02,967

## 5. 본 학습 1차: Linear + MLP

In [ ]:
%cd /content/lab2-problem

!python train.py --config configs/linear.yaml --opts dataset.num_workers=2
!python train.py --config configs/mlp_512.yaml --opts dataset.num_workers=2
!python train.py --config configs/mlp_2048.yaml --opts dataset.num_workers=2

## 6. 본 학습 2차: ConvNetV1 + ConvNetV2

In [ ]:
%cd /content/lab2-problem

!python train.py --config configs/convnet_v1.yaml --opts dataset.num_workers=2
!python train.py --config configs/convnet_v2.yaml --opts dataset.num_workers=2

## 7. 본 학습 3차: ConvNetV3-Reg + ConvNetV3-Aug

In [ ]:
%cd /content/lab2-problem

!python train.py --config configs/convnet_v3_reg.yaml --opts dataset.num_workers=2
!python train.py --config configs/convnet_v3_aug.yaml --opts dataset.num_workers=2

## 8. 본 학습 4차: ResNet50 Scratch + Fine-tune

In [ ]:
%cd /content/lab2-problem

!python train.py --config configs/resnet50_scratch.yaml --opts dataset.num_workers=2
!python train.py --config configs/resnet50_finetune.yaml --opts dataset.num_workers=2

## 9. 로그 확인

README Table 1에는 `train/loss`, `train/accuracy`, `val/loss`, `val/accuracy`, `params/trainable` 값을 옮기면 됩니다. 코드상 `val`은 CIFAR-10 test set 평가 결과입니다.

In [ ]:
%cd /content/lab2-problem

!find outputs -name "train.log" | sort
!tail -n 5 outputs/*/train.log

## 10. 결과 압축 다운로드

In [ ]:
%cd /content/lab2-problem

!zip -r outputs.zip outputs

from google.colab import files
files.download("outputs.zip")